# Historical research notebook

Sanitized source archive, not a validated current model. Saved outputs were removed. Data, credentials and model weights are not included. Path placeholders are not automatically expanded. See docs/CODE_CHANGELOG.md.


# ⚾ MLB 시즌 데이터 크롤링 (범용)

**사용법: `SEASON` 값만 바꾸고 전체 실행!**

수집 파일 (19개):
| # | 파일 | 소스 | 예상 시간 |
|---|---|---|---|
| 1-5 | 정규시즌 (경기결과/특이사항/피치/투수/타자) | MLB Stats API | ~60분 |
| 6 | 정규시즌 키플레이어 | MLB Stats API (topPerformers) | ~40분 |
| 7-11 | 시범경기 (경기결과/특이사항/피치/투수/타자) | MLB Stats API | ~10분 |
| 12 | 시범경기 키플레이어 | MLB Stats API (topPerformers) | ~5분 |
| 13 | 심판 | MLB Stats API (boxscore) | ~40분 |
| 14 | 팀순위 | MLB Stats API | 즉시 |
| 15 | 트랜잭션 | MLB Stats API | ~2분 |
| 16-17 | 선수시즌기록 (타자/투수) | MLB Stats API | 즉시 |
| 18-19 | Statcast (타자/투수) | Baseball Savant | 즉시 |

⏱️ 전체 약 2~3시간 (정규시즌 크롤링이 대부분)

## 🔧 Step 0. 설정
**⬇️ 여기서 연도만 바꾸세요!**

In [ ]:
# ============================================================
# ★★★ 이 값만 바꾸면 됩니다 ★★★
SEASON = 2026
# ============================================================

!pip install tqdm -q

from google.colab import drive
drive.mount('/content/drive')

import requests
import pandas as pd
import numpy as np
import time
import os
import json
from tqdm import tqdm
from datetime import datetime, timedelta
from io import StringIO

SAVE_DIR = f'<DRIVE_ROOT>/MLB데이터/{SEASON}'
os.makedirs(SAVE_DIR, exist_ok=True)
BASE_URL = 'https://statsapi.mlb.com/api/v1'

def mlb_api(url, retries=3, delay=0.3):
    for attempt in range(retries):
        try:
            r = requests.get(url, timeout=60)
            r.raise_for_status()
            time.sleep(delay)
            return r.json()
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(3)
            else:
                print(f"  ❌ 실패: {url[:80]}... - {e}")
                return None

print(f"✅ 설정 완료: MLB {SEASON} 시즌")
print(f"   저장 경로: {SAVE_DIR}")

## 📅 Step 1. 경기 목록

In [ ]:
def get_schedule(season, game_type):
    type_name = '정규시즌' if game_type == 'R' else '시범경기'
    print(f"📅 {season} {type_name} 경기 목록 조회 중...")
    url = f'{BASE_URL}/schedule?sportId=1&gameType={game_type}&startDate={season}-01-01&endDate={season}-12-31&hydrate=linescore,team,venue'
    data = mlb_api(url)
    if not data: return []
    games = []
    for date_info in data.get('dates', []):
        for game in date_info.get('games', []):
            if game.get('status', {}).get('detailedState', '') in ['Final', 'Completed Early']:
                games.append(game)
    print(f"   ✅ {len(games)}경기")
    return games

regular_games = get_schedule(SEASON, 'R')
spring_games = get_schedule(SEASON, 'S')

## 📊 Step 2. 경기결과

In [ ]:
def parse_game_results(games, series_name):
    rows = []
    for g in games:
        try:
            home = g['teams']['home']; away = g['teams']['away']
            ls = g.get('linescore', {})
            hs = home.get('score', 0); aws = away.get('score', 0)
            rows.append({
                'gamePk': g['gamePk'], '날짜': g['officialDate'],
                '홈팀': home['team']['name'], '홈팀ID': home['team']['id'],
                '원정팀': away['team']['name'], '원정팀ID': away['team']['id'],
                '구장': g.get('venue', {}).get('name', ''),
                '홈팀득점': hs, '원정팀득점': aws,
                '홈팀안타': ls.get('teams', {}).get('home', {}).get('hits', 0),
                '원정팀안타': ls.get('teams', {}).get('away', {}).get('hits', 0),
                '홈팀실책': ls.get('teams', {}).get('home', {}).get('errors', 0),
                '원정팀실책': ls.get('teams', {}).get('away', {}).get('errors', 0),
                '시즌': SEASON, '시리즈': series_name,
                '홈팀결과': '승' if hs > aws else ('패' if hs < aws else '무'),
                '원정팀결과': '승' if aws > hs else ('패' if aws < hs else '무'),
                '총득점': hs + aws, '점수차': abs(hs - aws),
            })
        except: pass
    return pd.DataFrame(rows)

for games, name in [(regular_games, '정규시즌'), (spring_games, '시범경기')]:
    df = parse_game_results(games, name)
    df.to_csv(f'{SAVE_DIR}/mlb_{SEASON}_{name}_경기결과.csv', index=False, encoding='utf-8-sig')
    print(f"✅ {name}_경기결과: {df.shape}")

## ⚙️ Step 3. 상세 크롤링 함수 정의

In [ ]:
# === 특이사항 ===
def get_notable_plays(game_pk, season):
    try:
        url = f"https://statsapi.mlb.com/api/v1/game/{game_pk}/playByPlay"
        data = requests.get(url, timeout=10).json()
        notable_events = [
            "Home Run", "Stolen Base", "Caught Stealing",
            "Error", "Double Play", "Triple Play",
            "Wild Pitch", "Passed Ball", "Balk",
            "Hit By Pitch", "Sac Fly", "Sac Bunt"
        ]
        plays = []
        for play in data.get("allPlays", []):
            event = play["result"].get("event", "")
            if event in notable_events:
                plays.append({
                    "gamePk": game_pk,
                    "이닝": play["about"]["inning"],
                    "공수": play["about"]["halfInning"],
                    "항목": event,
                    "타자": play["matchup"]["batter"]["fullName"],
                    "투수": play["matchup"]["pitcher"]["fullName"],
                    "내용": play["result"].get("description", ""),
                    "RBI": play["result"].get("rbi", 0),
                    "홈팀점수": play["result"].get("homeScore", 0),
                    "원정팀점수": play["result"].get("awayScore", 0),
                    "시즌": season,
                })
        return plays
    except: return []

# === 피치데이터 ===
def get_pitch_data(game_pk, season):
    try:
        url = f"https://statsapi.mlb.com/api/v1/game/{game_pk}/playByPlay"
        data = requests.get(url, timeout=10).json()
        pitches = []
        for play in data.get("allPlays", []):
            about = play.get("about", {}); matchup = play.get("matchup", {})
            at_bat_result = play.get("result", {}).get("event", "")
            for ev in play.get("playEvents", []):
                if not ev.get("isPitch", False): continue
                details = ev.get("details", {}); pd_ = ev.get("pitchData", {})
                count = ev.get("count", {}); brk = pd_.get("breaks", {})
                pt = details.get("type", {}).get("description", "")
                pc = details.get("type", {}).get("code", "")
                pitches.append({
                    "gamePk": game_pk, "이닝": about.get("inning", 0),
                    "공수": about.get("halfInning", ""),
                    "타자": matchup.get("batter", {}).get("fullName", ""),
                    "타자ID": matchup.get("batter", {}).get("id", ""),
                    "투수": matchup.get("pitcher", {}).get("fullName", ""),
                    "투수ID": matchup.get("pitcher", {}).get("id", ""),
                    "타석결과": at_bat_result, "피치결과": details.get("description", ""),
                    "구종": pt if pt else pc,
                    "구속": pd_.get("startSpeed", np.nan),
                    "회전수": pd_.get("spinRate", np.nan),
                    "수직무브": brk.get("breakVerticalInduced", brk.get("breakVertical", np.nan)),
                    "수평무브": brk.get("breakHorizontal", np.nan),
                    "존": pd_.get("zone", np.nan),
                    "볼카운트_볼": count.get("balls", 0),
                    "볼카운트_스트": count.get("strikes", 0),
                    "아웃수": count.get("outs", 0), "시즌": season,
                })
        return pitches
    except: return []

# === 타자/투수 개인기록 ===
def get_individual_stats(game_pk, season):
    try:
        url = f"https://statsapi.mlb.com/api/v1/game/{game_pk}/boxscore"
        data = requests.get(url, timeout=10).json()
        batters, pitchers = [], []
        for side in ["home", "away"]:
            team = data["teams"][side]; team_name = team["team"]["name"]
            for pid, player in team["players"].items():
                stats = player.get("stats", {})
                batting = stats.get("batting", {}); pitching = stats.get("pitching", {})
                ab = batting.get("atBats", 0); h = batting.get("hits", 0)
                bb = batting.get("baseOnBalls", 0); hbp = batting.get("hitByPitch", 0)
                sf = batting.get("sacFlies", 0); pa = batting.get("plateAppearances", 0)
                tb = batting.get("totalBases", 0)
                if ab > 0 or pa > 0:
                    avg = round(h/ab, 3) if ab > 0 else 0.0
                    obp = round((h+bb+hbp)/(ab+bb+hbp+sf), 3) if (ab+bb+hbp+sf) > 0 else 0.0
                    slg = round(tb/ab, 3) if ab > 0 else 0.0
                    batters.append({
                        "gamePk": game_pk, "팀": team_name, "홈원정": side,
                        "선수명": player["person"]["fullName"], "선수ID": player["person"]["id"],
                        "포지션": player.get("position", {}).get("abbreviation", ""),
                        "타석": pa, "타수": ab, "안타": h,
                        "2루타": batting.get("doubles", 0), "3루타": batting.get("triples", 0),
                        "홈런": batting.get("homeRuns", 0), "타점": batting.get("rbi", 0),
                        "득점": batting.get("runs", 0), "볼넷": bb,
                        "삼진": batting.get("strikeOuts", 0), "도루": batting.get("stolenBases", 0),
                        "타율": avg, "출루율": obp, "장타율": slg,
                        "OPS": round(obp + slg, 3), "시즌": season, "루타": tb,
                    })
                ip_str = pitching.get("inningsPitched", "0.0")
                if ip_str not in ["", "0.0", "0"]:
                    try:
                        parts = str(ip_str).split(".")
                        ip_val = int(parts[0]) + int(parts[1])/3 if len(parts) > 1 else float(ip_str)
                    except: ip_val = 0.0
                    er = pitching.get("earnedRuns", 0)
                    pit_h = pitching.get("hits", 0); pit_bb = pitching.get("baseOnBalls", 0)
                    pitchers.append({
                        "gamePk": game_pk, "팀": team_name, "홈원정": side,
                        "선수명": player["person"]["fullName"], "선수ID": player["person"]["id"],
                        "이닝": float(ip_str), "타자": pitching.get("battersFaced", 0),
                        "투구수": pitching.get("numberOfPitches", 0),
                        "피안타": pit_h, "홈런허용": pitching.get("homeRuns", 0),
                        "볼넷": pit_bb, "삼진": pitching.get("strikeOuts", 0),
                        "실점": pitching.get("runs", 0), "자책": er,
                        "ERA": round((er*9)/ip_val, 2) if ip_val > 0 else 0.0,
                        "WHIP": round((pit_h+pit_bb)/ip_val, 2) if ip_val > 0 else 0.0,
                        "승": pitching.get("wins", 0), "패": pitching.get("losses", 0),
                        "세이브": pitching.get("saves", 0), "홀드": pitching.get("holds", 0),
                        "시즌": season,
                    })
        return batters, pitchers
    except: return [], []

# === 키플레이어 ===
def get_key_players(game_pk, season):
    try:
        url = f"https://statsapi.mlb.com/api/v1/game/{game_pk}/boxscore"
        data = requests.get(url, timeout=10).json()
        players = []
        for performer in data.get("topPerformers", []):
            p = performer["player"]; stats = p["stats"]
            batting = stats.get("batting", {}); pitching = stats.get("pitching", {})
            players.append({
                "gamePk": game_pk,
                "선수명": p["person"]["fullName"], "선수ID": p["person"]["id"],
                "포지션": p["position"]["abbreviation"],
                "타입": performer.get("type", ""),
                "게임스코어": performer.get("gameScore", 0),
                "타수": batting.get("atBats", 0), "안타": batting.get("hits", 0),
                "홈런": batting.get("homeRuns", 0), "타점": batting.get("rbi", 0),
                "득점": batting.get("runs", 0),
                "이닝": pitching.get("inningsPitched", None),
                "투구수": pitching.get("numberOfPitches", 0),
                "삼진": pitching.get("strikeOuts", 0),
                "자책": pitching.get("earnedRuns", 0),
                "요약": pitching.get("summary", None),
            })
        return players
    except: return []

# === 심판 ===
def get_umpires(game_pk, season, date):
    try:
        url = f"https://statsapi.mlb.com/api/v1/game/{game_pk}/boxscore"
        data = requests.get(url, timeout=10).json()
        return [{"gamePk": game_pk, "날짜": date, "시즌": season,
                 "심판명": u["official"]["fullName"], "심판ID": u["official"]["id"],
                 "포지션": u["officialType"]} for u in data.get("officials", [])]
    except: return []

print("✅ 함수 정의 완료")

## 🔄 Step 4. 정규시즌 상세 크롤링
⏱️ ~2400경기, 약 **60분** 소요 (100경기마다 체크포인트)

In [ ]:
def crawl_games(games, series_name, save_dir, season):
    df_results = pd.read_csv(f'{save_dir}/mlb_{season}_{series_name}_경기결과.csv')
    date_map = dict(zip(df_results['gamePk'], df_results['날짜']))
    game_pks = df_results['gamePk'].tolist()

    # 체크포인트
    cp_file = f'{save_dir}/_checkpoint_{series_name}.json'
    done = set()
    if os.path.exists(cp_file):
        with open(cp_file) as f: done = set(json.load(f))
        print(f"  📂 체크포인트: {len(done)}개 완료")

    remaining = [pk for pk in game_pks if pk not in done]
    print(f"\n🔄 {series_name} 크롤링 ({len(game_pks)}경기, 남은 {len(remaining)}개)")

    all_특이, all_피치, all_타자, all_투수 = [], [], [], []

    for i, gpk in enumerate(tqdm(remaining, desc=f'{series_name}')):
        all_특이.extend(get_notable_plays(gpk, season))
        all_피치.extend(get_pitch_data(gpk, season))
        bat, pit = get_individual_stats(gpk, season)
        all_타자.extend(bat); all_투수.extend(pit)
        done.add(gpk)

        if (i+1) % 100 == 0:
            print(f"\n  💾 체크포인트 ({len(done)}/{len(game_pks)})")
            with open(cp_file, 'w') as f: json.dump(list(done), f)
            for nm, lst in [('특이사항', all_특이), ('피치데이터', all_피치),
                            ('투수개인기록', all_투수), ('타자개인기록', all_타자)]:
                pd.DataFrame(lst).to_csv(f'{save_dir}/_temp_{series_name}_{nm}.csv',
                                         index=False, encoding='utf-8-sig')

    # 최종 저장
    prefix = f'mlb_{season}_{series_name}'
    for nm, lst in [('특이사항', all_특이), ('피치데이터', all_피치),
                    ('투수개인기록', all_투수), ('타자개인기록', all_타자)]:
        # 기존 체크포인트 데이터 병합
        temp = f'{save_dir}/_temp_{series_name}_{nm}.csv'
        if os.path.exists(temp) and len(done) > len(remaining):
            old = pd.read_csv(temp)
            df = pd.concat([old, pd.DataFrame(lst)], ignore_index=True)
        else:
            df = pd.DataFrame(lst)
        df.to_csv(f'{save_dir}/{prefix}_{nm}.csv', index=False, encoding='utf-8-sig')
        print(f"  ✅ {prefix}_{nm}: {df.shape}")

    # 임시파일 정리
    for f in os.listdir(save_dir):
        if f.startswith('_temp_') or f.startswith('_checkpoint_'):
            os.remove(f'{save_dir}/{f}')

crawl_games(regular_games, '정규시즌', SAVE_DIR, SEASON)

## 🌱 Step 5. 시범경기 상세 크롤링
⏱️ 약 **10분**

In [ ]:
crawl_games(spring_games, '시범경기', SAVE_DIR, SEASON)

## 🌟 Step 6. 키플레이어
⏱️ 약 **40분** (정규시즌) + **5분** (시범경기)

In [ ]:
kp_col_order = ['gamePk', '날짜', '팀', '선수명', '선수ID', '포지션', '타입',
                '게임스코어', '타수', '안타', '홈런', '타점', '득점',
                '이닝', '투구수', '삼진', '자책', '요약', '시즌']

for series in ['정규시즌', '시범경기']:
    df_results = pd.read_csv(f'{SAVE_DIR}/mlb_{SEASON}_{series}_경기결과.csv')
    date_map = dict(zip(df_results['gamePk'], df_results['날짜']))
    game_pks = df_results['gamePk'].tolist()

    df_bat = pd.read_csv(f'{SAVE_DIR}/mlb_{SEASON}_{series}_타자개인기록.csv')
    df_pit = pd.read_csv(f'{SAVE_DIR}/mlb_{SEASON}_{series}_투수개인기록.csv')
    player_team = {}
    for _, r in df_bat.iterrows(): player_team[(r['gamePk'], r['선수ID'])] = r['팀']
    for _, r in df_pit.iterrows(): player_team[(r['gamePk'], r['선수ID'])] = r['팀']

    all_kp = []
    print(f"\n🔄 {series} 키플레이어 ({len(game_pks)}경기)...")
    for gpk in tqdm(game_pks, desc=f'{series} 키플레이어'):
        kps = get_key_players(gpk, SEASON)
        for k in kps:
            k['날짜'] = date_map.get(gpk, '')
            k['팀'] = player_team.get((gpk, k['선수ID']), '')
            k['시즌'] = SEASON
        all_kp.extend(kps)
        time.sleep(0.3)

    df = pd.DataFrame(all_kp)[kp_col_order]
    df.to_csv(f'{SAVE_DIR}/mlb_{SEASON}_{series}_키플레이어.csv', index=False, encoding='utf-8-sig')
    print(f"✅ {series}_키플레이어: {df.shape}")

## 👨‍⚖️ Step 7. 심판
⏱️ 약 **40분**

In [ ]:
ump_col_order = ['gamePk', '날짜', '시즌', '심판명', '심판ID', '포지션']
all_umpires = []

for series in ['정규시즌', '시범경기']:
    df_results = pd.read_csv(f'{SAVE_DIR}/mlb_{SEASON}_{series}_경기결과.csv')
    date_map = dict(zip(df_results['gamePk'], df_results['날짜']))
    game_pks = df_results['gamePk'].tolist()

    print(f"\n🔄 {series} 심판 ({len(game_pks)}경기)...")
    for gpk in tqdm(game_pks, desc=f'{series} 심판'):
        all_umpires.extend(get_umpires(gpk, SEASON, date_map.get(gpk, '')))
        time.sleep(0.3)

df = pd.DataFrame(all_umpires)[ump_col_order]
df.to_csv(f'{SAVE_DIR}/mlb_{SEASON}_심판.csv', index=False, encoding='utf-8-sig')
print(f"\n✅ 심판: {df.shape}")

## 🏆 Step 8. 팀순위

In [ ]:
def get_standings(season):
    url = "https://statsapi.mlb.com/api/v1/standings"
    params = {"leagueId": "103,104", "season": season, "standingsTypes": "regularSeason"}
    data = requests.get(url, params=params, timeout=10).json()
    rows = []
    for record in data.get("records", []):
        for tr in record.get("teamRecords", []):
            wp = tr.get("winningPercentage", "")
            try: wp_float = float(wp)
            except: wp_float = 0.0
            rows.append({
                "시즌": season, "팀명": tr["team"]["name"], "팀ID": tr["team"]["id"],
                "디비전순위": int(tr.get("divisionRank", 0)),
                "리그순위": int(tr.get("leagueRank", 0)),
                "경기수": tr.get("gamesPlayed", 0),
                "승": tr.get("wins", 0), "패": tr.get("losses", 0),
                "승률": wp_float, "게임차": tr.get("gamesBack", "-"),
                "득점": tr.get("runsScored", 0), "실점": tr.get("runsAllowed", 0),
                "득실차": tr.get("runDifferential", 0),
                "와일드카드게임차": tr.get("wildCardGamesBack", "-"),
            })
    return pd.DataFrame(rows)

df = get_standings(SEASON)
df.to_csv(f'{SAVE_DIR}/mlb_{SEASON}_팀순위.csv', index=False, encoding='utf-8-sig')
print(f"✅ 팀순위: {df.shape}")

## 📋 Step 9. 트랜잭션
⏱️ 365일 순회, 약 **2분**

In [ ]:
def get_transactions(date, season):
    try:
        url = "https://statsapi.mlb.com/api/v1/transactions"
        params = {"startDate": date, "endDate": date, "sportId": 1}
        data = requests.get(url, params=params, timeout=10).json()
        rows = []
        for t in data.get("transactions", []):
            month = int(date.split("-")[1])
            person = t.get("person", {}); to_team = t.get("toTeam", {})
            rows.append({
                "시즌": season, "월": month, "날짜": t.get("date", ""),
                "선수명": person.get("fullName", None) if person else None,
                "선수ID": person.get("id", None) if person else None,
                "팀명": to_team.get("name", "") if to_team else "",
                "팀ID": to_team.get("id", 0) if to_team else 0,
                "유형코드": t.get("typeCode", ""), "유형": t.get("typeDesc", ""),
                "내용": t.get("description", ""),
            })
        return rows
    except: return []

start = datetime(SEASON, 1, 1); end = datetime(SEASON, 12, 31)
dates = []
d = start
while d <= end: dates.append(d.strftime("%Y-%m-%d")); d += timedelta(days=1)

all_txn = []
for date in tqdm(dates, desc='트랜잭션'):
    all_txn.extend(get_transactions(date, SEASON))
    time.sleep(0.2)

df = pd.DataFrame(all_txn)
df.to_csv(f'{SAVE_DIR}/mlb_{SEASON}_트랜잭션.csv', index=False, encoding='utf-8-sig')
print(f"✅ 트랜잭션: {df.shape}")

## ⚾ Step 10. 선수 시즌 기록

In [ ]:
def get_season_stats(season, group):
    url = "https://statsapi.mlb.com/api/v1/stats"
    all_rows = []; offset = 0
    while True:
        params = {"stats": "season", "group": group, "season": season,
                  "sportId": 1, "playerPool": "ALL", "limit": 500, "offset": offset}
        data = requests.get(url, params=params, timeout=30).json()
        splits = data.get("stats", [{}])[0].get("splits", [])
        if not splits: break
        for s in splits:
            row = {"시즌": season, "구분": group}
            row["선수명"] = s.get("player", {}).get("fullName", "")
            row["선수ID"] = s.get("player", {}).get("id", 0)
            row["팀명"] = s.get("team", {}).get("name", "")
            row["팀ID"] = s.get("team", {}).get("id", 0)
            row.update(s.get("stat", {}))
            all_rows.append(row)
        offset += len(splits)
        print(f"  {group}: {offset}명")
        if len(splits) < 500: break
    return pd.DataFrame(all_rows)

for group, label in [("hitting", "타자"), ("pitching", "투수")]:
    df = get_season_stats(SEASON, group)
    df.to_csv(f'{SAVE_DIR}/mlb_{SEASON}_선수시즌기록_{label}.csv', index=False, encoding='utf-8-sig')
    print(f"✅ 선수시즌기록_{label}: {df.shape}")

## 📈 Step 11. Statcast

In [ ]:
def parse_name(name_str):
    try:
        parts = name_str.split(", ")
        return f"{parts[1]} {parts[0]}"
    except: return name_str

base_url = "https://baseballsavant.mlb.com/leaderboard/expected_statistics"
ev_url = "https://baseballsavant.mlb.com/leaderboard/statcast"

for player_type, label in [("pitcher", "투수"), ("batter", "타자")]:
    print(f"\n{label} 수집 중...")
    r = requests.get(base_url, params={"type":player_type,"year":str(SEASON),"min":"q","csv":"true"}, timeout=30)
    df_x = pd.read_csv(StringIO(r.text))
    df_x["선수명"] = df_x["last_name, first_name"].apply(parse_name)
    df_x["시즌"] = SEASON; df_x["구분"] = label
    print(f"  xstats: {len(df_x)}명")

    r = requests.get(ev_url, params={"type":player_type,"year":str(SEASON),"min":"q","csv":"true"}, timeout=30)
    df_ev = pd.read_csv(StringIO(r.text))
    df_ev["선수명"] = df_ev["last_name, first_name"].apply(parse_name)

    df = df_x.merge(
        df_ev[["player_id","avg_hit_speed","ev50","barrels","brl_percent","brl_pa","선수명"]],
        on=["player_id","선수명"], how="left"
    )
    df.to_csv(f'{SAVE_DIR}/mlb_{SEASON}_statcast_{label}.csv', index=False, encoding='utf-8-sig')
    print(f"✅ statcast_{label}: {df.shape}")

## ✅ Step 12. 최종 확인

In [ ]:
print("=" * 60)
print(f"🎉 MLB {SEASON} 데이터 크롤링 완료!")
print("=" * 60)

for f in sorted(os.listdir(SAVE_DIR)):
    if f.endswith('.csv') and not f.startswith('_'):
        size = os.path.getsize(f'{SAVE_DIR}/{f}') / 1024
        print(f"  📄 {f:50s} ({size:>10.1f}KB)")

print(f"\n📁 저장 경로: {SAVE_DIR}")
print(f"\n총 {len([f for f in os.listdir(SAVE_DIR) if f.endswith('.csv') and not f.startswith('_')])}개 파일")